# P37 — MemGPT: modelos de lenguaje como sistemas operativos

## 1. Título y paper

**Paper:** *MemGPT: Towards LLMs as Operating Systems*  
**Autoría:** Charles Packer, Sarah Wooders, Kevin Lin, Vivian Fang, Shishir G. Patil, Ion Stoica, Joseph E. Gonzalez  
**Año y venue:** 2023 · arXiv:2310.08560  
**Nivel:** L3 · **Motor:** `memgpt`  
**Ficha completa:** [`P37_memgpt`](../../papers/foundational/P37_memgpt/README.md)

**Hito:** Aplica al contexto la idea de memoria virtual: una jerarquía que da la ilusión de memoria grande sobre una pequeña y rápida.

- [arXiv:2310.08560](https://arxiv.org/abs/2310.08560)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: La ventana de contexto es un límite duro. Ampliarla es caro y, como muestra P36, no garantiza que se use bien.
2. Ejecutar una implementación mínima de la propuesta: Gestionar el contexto como un sistema operativo gestiona la memoria: un contexto principal pequeño, un almacén externo grande, y el propio modelo decidiendo qué paginar mediante llamadas de función.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P11
- P31
- P36


## 4. Intuición

Tu ordenador te deja abrir archivos mucho más grandes que su memoria RAM. No los carga enteros: los pagina. MemGPT hace lo mismo con el contexto — y quien decide qué paginar es el propio modelo.


## 5. Concepto mínimo

```text
contexto principal   (pequeño, rápido, siempre visible)   ← como la RAM
almacén externo      (grande, lento, accesible por función) ← como el disco

    el modelo llama a funciones para mover información entre ambos
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('memgpt', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Qué pasa con el sexto dato si la capacidad es de cinco?
2. ¿Se pierde?
3. ¿Qué cuesta recuperarlo?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('memgpt', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('memgpt', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Lo desalojado **no se pierde**: baja al almacén externo y vuelve con una llamada de función. La ilusión es de memoria grande; la realidad es una jerarquía con coste por acceso.


## 10. Comentario pedagógico

La analogía con el sistema operativo es literal y ese es su valor: da un vocabulario prestado —paginación, jerarquía, interrupciones— para un problema que se estaba tratando sin marco.


## 11. Error o anti-patrón deliberado

Anti-patrón: tratar el almacén externo como gratis.


In [ ]:
for consultas in (1, 10, 100):
    print(f'{consultas:>3} page-ins → {consultas} llamadas extra al modelo '
          f'({consultas * 2:>3} segundos aprox. de latencia añadida)')

## 12. Corrección

Lo correcto es presupuestar los accesos igual que cualquier otra llamada:


In [ ]:
presupuesto = {'page_ins_maximos_por_respuesta': 3,
               'que_va_al_contexto_principal': 'lo que se usa en casi toda interaccion',
               'que_va_al_externo': 'historico, detalles puntuales, documentos',
               'si_se_agota': 'responder con lo que hay y declarar la limitacion'}
show(presupuesto)

## 13. Desafío guiado

Reduce la capacidad del contexto principal a 2 y comprueba cuántos datos acaban en el almacén externo.


In [ ]:
r = run_paper_lab('memgpt', seed=3)['result']
show(r)

## 14. Desafío autónomo

Implementa una jerarquía de memoria de dos niveles para un asistente propio, con política de desalojo explícita. Mide cuántas veces hay que paginar en 50 conversaciones reales.


## 15. Evidencia de aprendizaje

Guarda la traza de desalojos y recuperaciones, y tu presupuesto de accesos por respuesta.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P37_memgpt/README.md) · evaluación formal: [`assessments/papers/P37_memgpt.md`](../../assessments/papers/P37_memgpt.md)


## 16. Cierre

Con esto se cierra el bloque de memoria y contexto. Lo que sigue es el andamiaje que hace entrenable todo lo anterior.


## 17. Conexión con el siguiente hito

- P16
- gestión de contexto en agentes

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
